# Text Preprocessing Pipeline
## FYE Town Hall Meeting Survey Analysis

**Author:** Student Researcher  
**Project:** First Year Experience Research and Engagement  
**Institution:** California State University, Chico  
**Date:** March 2026

---

## Notebook Overview

This notebook implements the text preprocessing pipeline for THM survey responses across 7 semesters (Fall 2022 - Fall 2025). 

### Objectives:
1. Load cleaned survey data from Google Drive
2. Implement hybrid text preprocessing strategy:
   - **Token-level cleaning** for topic modeling and keyword extraction
   - **Sentence-level preservation** for sentiment analysis
3. Create separate corpora by Question_Category
4. Save preprocessed data back to Google Drive
5. Generate preprocessing statistics and validation samples

---
## 1. Environment Setup

In [2]:
import sys
import re
import string
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append('..')
from utils.gdrive_auth import DriveDataLoader
from config import PREPROCESSED_DATA_FOLDER_ID, MAIN_FOLDER_ID, GDRIVE_FILES

print("✓ Imports successful")

✓ Imports successful


### Download Required NLTK Resources

In [3]:
required_nltk = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4']

print("Downloading NLTK resources...")
for resource in required_nltk:
    try:
        nltk.download(resource, quiet=True)
        print(f"  ✓ {resource}")
    except Exception as e:
        print(f"  ✗ {resource}: {e}")

print("\nNLTK setup complete!")

  ✓ punkt
  ✓ punkt_tab
  ✓ stopwords
  ✓ wordnet
  ✓ averaged_perceptron_tagger
  ✓ omw-1.4

NLTK setup complete!


---
## 2. Load Data from Google Drive

In [4]:
print("Initializing Google Drive connection...")
loader = DriveDataLoader()

all_ok = loader.run_full_check(
    main_folder_id   = MAIN_FOLDER_ID,
    output_folder_id = PREPROCESSED_DATA_FOLDER_ID,
    gdrive_files     = GDRIVE_FILES
)

if not all_ok:
    raise SystemExit('\n⛔ Access check failed. Fix the issues above before proceeding.')

Initializing Google Drive connection...

  FYE PROJECT — GOOGLE DRIVE ACCESS CHECK

GOOGLE DRIVE AUTHENTICATION

🔐 You will be prompted to log in with your Google account
📋 Make sure you have access to the THM_Survey_Data folder

Steps:
  1. Browser will open
  2. Select your Google account
  3. Click 'Allow' to grant access
  4. Return here to continue


→ Using saved credentials...

✓ AUTHENTICATED AS: hvhasabnis@csuchico.edu

✓ Ready to load data from Google Drive

  🔍 Checking READ access... ✅  (2 files visible)
  🔍 Checking WRITE access... ✅

  🔍 Verifying all dataset files...
    ✅  text_prepared        — THM_All_Semesters_TEXT_PREPARED (315 KB)
    ✅  quant_prepared       — THM_All_Semesters_QUANTITATIVE_PREPARED (17 KB)
    ✅  fall_2022            — THM_Fall_2022 (60 KB)
    ✅  spring_2023          — THM_Spring_2023 (63 KB)
    ✅  fall_2023            — THM_Fall_2023 (33 KB)
    ✅  spring_2024          — THM_Spring_2024 (23 KB)
    ✅  fall_2024            — THM_Fall_2024 (61 KB

In [5]:
print("Loading merged THM dataset from Drive...")
df = loader.read_csv(GDRIVE_FILES['text_prepared'],  label='text_prepared')

print(f"\nDataset loaded successfully!")
print(f"  Total records: {len(df):,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Semesters: {df['Semester'].nunique()}")

Loading merged THM dataset from Drive...
  [text_prepared] Loading CSV (ID: 1OYdTNbBJNsD0_fHLyQ6...)... ✓ (3585, 18)

Dataset loaded successfully!
  Total records: 3,585
  Columns: 18
  Semesters: 7


### Identify Text Columns

In [6]:
exclude_cols = ['Semester', 'Timestamp', 'Question_Category', 'Response_ID']
text_columns = [col for col in df.columns if col not in exclude_cols and df[col].dtype == 'object']

print(f"Found {len(text_columns)} text response columns:")
for i, col in enumerate(text_columns, 1):
    non_null = df[col].notna().sum()
    print(f"  {i}. {col}: {non_null:,} responses")

Found 9 text response columns:
  1. ResponseID: 3,585 responses
  2. Term: 3,585 responses
  3. PolicyArea: 883 responses
  4. Question_Column: 3,585 responses
  5. Question_Short: 3,585 responses
  6. Response_Text: 3,585 responses
  7. Response_Text_Clean: 3,585 responses
  8. Dataset: 3,585 responses
  9. Date_Extracted: 3,585 responses


---
## 3. Initial Data Exploration

In [7]:
def combine_text_responses(row, text_cols):
    responses = []
    for col in text_cols:
        if pd.notna(row[col]) and str(row[col]).strip():
            responses.append(str(row[col]).strip())
    return ' | '.join(responses) if responses else None

df['combined_text'] = df.apply(lambda row: combine_text_responses(row, text_columns), axis=1)
df_text = df[df['combined_text'].notna()].copy()

print(f"Text responses available: {len(df_text):,} records")

Text responses available: 3,585 records


In [8]:
df_text['text_length'] = df_text['combined_text'].str.len()
df_text['word_count'] = df_text['combined_text'].str.split().str.len()

print("\nTEXT STATISTICS")
print("="*60)
print(f"\nMean length: {df_text['text_length'].mean():.0f} characters")
print(f"Mean words: {df_text['word_count'].mean():.1f} words")
print(f"Median words: {df_text['word_count'].median():.0f} words")


TEXT STATISTICS

Mean length: 601 characters
Mean words: 106.3 words
Median words: 100 words


---
## 4. Text Preprocessing Functions

### 4.1 Basic Cleaning

In [9]:
def basic_clean(text):
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'([!?.])\1{2,}', r'\1', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

test = "Check https://example.com!!! test@email.com"
print("Test:", basic_clean(test))

Test: check


### 4.2 Sentence-Level Processing

In [10]:
def sentence_level_process(text):
    if not text:
        return []
    
    sentences = sent_tokenize(text)
    processed = []
    
    for sent in sentences:
        sent = re.sub(r'\s+', ' ', sent).strip()
        if len(sent.split()) > 2:
            processed.append(sent)
    
    return processed

test = "I love the campus! Professors are helpful."
print("Sentences:", sentence_level_process(test))

Sentences: ['I love the campus!', 'Professors are helpful.']


### 4.3 Token-Level Processing

In [11]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

preserve_words = {'no', 'not', 'more', 'most', 'very', 'can', 'should', 'would', 'need'}
stop_words = stop_words - preserve_words

print(f"Stopwords: {len(stop_words)} words")

Stopwords: 191 words


In [12]:
def token_level_process(text, remove_stopwords=True, lemmatize=True):
    if not text:
        return []
    
    tokens = word_tokenize(text)
    processed = []
    
    for token in tokens:
        token = token.translate(str.maketrans('', '', string.punctuation))
        
        if not token or len(token) < 3:
            continue
        
        if remove_stopwords and token.lower() in stop_words:
            continue
        
        if lemmatize:
            token = lemmatizer.lemmatize(token.lower())
        else:
            token = token.lower()
        
        processed.append(token)
    
    return processed

test = "The professors are very helpful!"
print("Tokens:", token_level_process(test))

Tokens: ['professor', 'very', 'helpful']


---
## 5. Apply Preprocessing Pipeline

In [13]:
print("Applying preprocessing pipeline...\n")

print("[1/3] Basic cleaning...")
df_text['text_cleaned'] = df_text['combined_text'].progress_apply(basic_clean)

print("\n[2/3] Sentence-level processing...")
df_text['sentences'] = df_text['text_cleaned'].progress_apply(sentence_level_process)
df_text['sentence_count'] = df_text['sentences'].apply(len)

print("\n[3/3] Token-level processing...")
df_text['tokens'] = df_text['text_cleaned'].progress_apply(
    lambda x: token_level_process(x, remove_stopwords=True, lemmatize=True)
)
df_text['token_count'] = df_text['tokens'].apply(len)

print("\n✓ Preprocessing complete!")

Applying preprocessing pipeline...

[1/3] Basic cleaning...


100%|██████████| 3585/3585 [00:00<00:00, 17079.74it/s]



[2/3] Sentence-level processing...


100%|██████████| 3585/3585 [00:00<00:00, 11499.29it/s]



[3/3] Token-level processing...


100%|██████████| 3585/3585 [00:02<00:00, 1390.11it/s]


✓ Preprocessing complete!


---
## 6. Preprocessing Statistics

In [14]:
print("\nPREPROCESSING STATISTICS")
print("="*60)

print("\nToken Reduction:")
print(f"  Original: {df_text['word_count'].mean():.1f} words")
print(f"  Processed: {df_text['token_count'].mean():.1f} tokens")
print(f"  Reduction: {(1 - df_text['token_count'].mean() / df_text['word_count'].mean()) * 100:.1f}%")

all_tokens = [t for tokens in df_text['tokens'] if tokens for t in tokens]
unique_tokens = set(all_tokens)

print("\nVocabulary:")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Unique tokens: {len(unique_tokens):,}")


PREPROCESSING STATISTICS

Token Reduction:
  Original: 106.3 words
  Processed: 51.9 tokens
  Reduction: 51.2%

Vocabulary:
  Total tokens: 185,906
  Unique tokens: 4,929


In [15]:
from collections import Counter

token_freq = Counter(all_tokens)
top_30 = token_freq.most_common(30)

print("\nTOP 30 TOKENS")
print("="*60)
for i, (token, freq) in enumerate(top_30, 1):
    print(f"{i:2d}. {token:20s} : {freq:5d}")


TOP 30 TOKENS
 1. town                 :  4216
 2. hall                 :  4196
 3. policy               :  3830
 4. describe             :  3798
 5. thmallsemesters      :  3585
 6. 20260211             :  3585
 7. 154400               :  3585
 8. event                :  3518
 9. meeting              :  3152
10. not                  :  2632
11. help                 :  2566
12. fall                 :  2510
13. question             :  2332
14. community            :  2255
15. accurately           :  2212
16. response             :  2210
17. feel                 :  2188
18. experience           :  2162
19. student              :  2112
20. confidence           :  2017
21. more                 :  2006
22. talking              :  1950
23. people               :  1808
24. change               :  1782
25. can                  :  1752
26. topic                :  1747
27. increase             :  1688
28. friend               :  1652
29. chico                :  1637
30. connection           :  

---
## 7. Validation Samples

In [16]:
print("\nBEFORE/AFTER SAMPLES")
print("="*60)

for idx in df_text.sample(3, random_state=42).index:
    print(f"\nORIGINAL: {df_text.loc[idx, 'combined_text'][:100]}...")
    print(f"TOKENS: {' '.join(df_text.loc[idx, 'tokens'][:15])}...")
    print("-" * 60)


BEFORE/AFTER SAMPLES

ORIGINAL: Fall_2025_116 | Fall | Academic Ability: Responses to this question help us accurately describe any ...
TOKENS: fall2025116 fall academic ability response question help accurately describe increase confidence student not yet participated...
------------------------------------------------------------

ORIGINAL: Fall_2025_80 | Fall | Academic Ability: Responses to this question help us accurately describe any i...
TOKENS: fall202580 fall academic ability response question help accurately describe increase confidence student not yet participated...
------------------------------------------------------------

ORIGINAL: Spring_2023_254 | Spring | What did the Town Hall Meeting mean to you? | What did the Town Hall Meet...
TOKENS: spring2023254 spring town hall meeting mean town hall meeting mean got motivation understood important engage...
------------------------------------------------------------


---
## 8. Save Preprocessed Data

In [17]:
df_full = df_text[[
    'Semester', 'Question_Category', 'combined_text', 'text_cleaned',
    'sentences', 'sentence_count', 'tokens', 'token_count'
]].copy()

df_sentences = df_text[['Semester', 'Question_Category', 'sentences']].copy()
df_sentences = df_sentences.explode('sentences').reset_index(drop=True)
df_sentences = df_sentences.rename(columns={'sentences': 'sentence'})
df_sentences = df_sentences[df_sentences['sentence'].notna()]

df_tokens = df_text[['Semester', 'Question_Category', 'tokens']].copy()
df_tokens['tokens_text'] = df_tokens['tokens'].apply(lambda x: ' '.join(x) if x else '')
df_tokens = df_tokens[df_tokens['tokens_text'] != '']

print(f"Full dataset: {len(df_full):,} rows")
print(f"Sentences: {len(df_sentences):,} sentences")
print(f"Tokens: {len(df_tokens):,} documents")

Full dataset: 3,585 rows
Sentences: 14,836 sentences
Tokens: 3,585 documents


In [18]:
output_dir = Path('../outputs/01_preprocessing')
output_dir.mkdir(parents=True, exist_ok=True)

df_full.to_csv(output_dir / 'preprocessed_full.csv', index=False)
df_sentences.to_csv(output_dir / 'preprocessed_sentences.csv', index=False)
df_tokens.to_csv(output_dir / 'preprocessed_tokens.csv', index=False)

print("✓ Files saved locally")

✓ Files saved locally


In [19]:
print("\nUploading to Google Drive...\n")

if loader._write_ok:
    files = [
        ('preprocessed_full.csv', 'Full dataset'),
        ('preprocessed_sentences.csv', 'Sentences'),
        ('preprocessed_tokens.csv', 'Tokens')
    ]
    
    for filename, desc in files:
        filepath = output_dir / filename
        existing_id = loader._get_existing_file_id(filename, PREPROCESSED_DATA_FOLDER_ID)
        
        if existing_id:
            file = loader.drive.CreateFile({'id': existing_id})
        else:
            file = loader.drive.CreateFile({
                'title': filename,
                'parents': [{'id': PREPROCESSED_DATA_FOLDER_ID}]
            })
        
        file.SetContentFile(str(filepath))
        file.Upload()
        print(f"✓ {desc}")
    
    print("\n✓ Upload complete!")
else:
    print("⚠ Write access unavailable")


Uploading to Google Drive...

✓ Full dataset
✓ Sentences
✓ Tokens

✓ Upload complete!


In [20]:
import shutil
import os
import time
import gc

print("\n🗑️  Cleaning up local files...")

def delete_local_data():
    """
    Delete outputs folder with retry logic for locked files.
    """
    # Force garbage collection to release any file handles
    gc.collect()
    
    folder = '../outputs'
    
    if not os.path.exists(folder):
        print(f"    ⚠️  Folder not found (already clean): {folder}/")
        return
    
    # Try deleting with retries
    max_retries = 3
    for attempt in range(max_retries):
        try:
            shutil.rmtree(folder)
            print(f"    ✅ Deleted local folder: {folder}/")
            break
        except PermissionError as e:
            if attempt < max_retries - 1:
                print(f"    ⚠️  Files locked, retrying in 2 seconds... (attempt {attempt + 1}/{max_retries})")
                time.sleep(2)
                gc.collect()  # Try to release handles again
            else:
                print(f"    ❌ Could not delete folder - files are locked")
                print(f"    💡 Close any open CSV/Excel files and try again")
                print(f"    📁 Locked file: {e.filename}")
                return

    print("\n" + "=" * 65)
    print("  ✅ ALL DONE — outputs saved to Drive, local files removed.")
    print("=" * 65)

delete_local_data()


🗑️  Cleaning up local files...
    ⚠️  Files locked, retrying in 2 seconds... (attempt 1/3)
    ⚠️  Files locked, retrying in 2 seconds... (attempt 2/3)
    ❌ Could not delete folder - files are locked
    💡 Close any open CSV/Excel files and try again
    📁 Locked file: ../outputs\01_preprocessing\preprocessed_tokens.csv
